In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [12]:
events = pd.DataFrame({
    'event_id': [1, 2, 3, 4, 5, 6, 7],
    'ten_su_kien': [
        'Hiến máu tình nguyện',
        'Hội thảo kỹ nă   ng mềm',
        'Tình nguyện dọn vệ sinh môi trường',
        'Cuộc thi lập trình ACM',
        'Hội thảo định hướng nghề nghiệp IT',
        'Chiến dịch hiến máu trường đại học',
        'Workshop kỹ năng thuyết trình'
    ],
    'danh_muc': [
        'tinh_nguyen', 'ky_nang', 'tinh_nguyen',
        'hoc_thuat', 'huong_nghiep', 'tinh_nguyen', 'ky_nang'
    ],
    'mo_ta': [
        'hiến máu tình nguyện nhân đạo cộng đồng sức khỏe',
        'kỹ năng mềm giao tiếp làm việc nhóm thuyết trình',
        'tình nguyện môi trường dọn dẹp cộng đồng xanh sạch',
        'lập trình thi đấu thuật toán công nghệ IT code',
        'nghề nghiệp IT công nghệ hướng dẫn định hướng việc làm',
        'hiến máu nhân đạo tình nguyện sức khỏe cộng đồng',
        'thuyết trình kỹ năng mềm giao tiếp tự tin trình bày'
    ],
    'diem_ren_luyen': [10, 5, 8, 7, 5, 10, 6]
})

print(events[['event_id', 'ten_su_kien', 'danh_muc', 'diem_ren_luyen']])

   event_id                         ten_su_kien      danh_muc  diem_ren_luyen
0         1                Hiến máu tình nguyện   tinh_nguyen              10
1         2                Hội thảo kỹ năng mềm       ky_nang               5
2         3  Tình nguyện dọn vệ sinh môi trường   tinh_nguyen               8
3         4              Cuộc thi lập trình ACM     hoc_thuat               7
4         5  Hội thảo định hướng nghề nghiệp IT  huong_nghiep               5
5         6  Chiến dịch hiến máu trường đại học   tinh_nguyen              10
6         7       Workshop kỹ năng thuyết trình       ky_nang               6


In [13]:
# Tạo TF-IDF matrix từ cột mô tả
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(events['mo_ta'])

# Tính cosine similarity giữa tất cả các sự kiện
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print("Shape của TF-IDF matrix:", tfidf_matrix.shape)
print("\nMa trận Cosine Similarity (làm tròn 2 chữ số):")
print(pd.DataFrame(cosine_sim).round(2))

Shape của TF-IDF matrix: (7, 43)

Ma trận Cosine Similarity (làm tròn 2 chữ số):
      0     1     2     3     4     5     6
0  1.00  0.00  0.29  0.00  0.00  1.00  0.00
1  0.00  1.00  0.00  0.06  0.15  0.00  0.64
2  0.29  0.00  1.00  0.00  0.00  0.29  0.00
3  0.00  0.06  0.00  1.00  0.21  0.00  0.11
4  0.00  0.15  0.00  0.21  1.00  0.00  0.00
5  1.00  0.00  0.29  0.00  0.00  1.00  0.00
6  0.00  0.64  0.00  0.11  0.00  0.00  1.00


In [17]:
# Lịch sử đăng ký của sinh viên
lich_su_dang_ky = {
    'SV001': [1, 6],        # đã đăng ký event_id 1 và 6
    'SV002': [1, 3, 6],
    'SV003': [2, 4, 5, 7],
}

def goi_y_content_based_cho_user(student_id, top_n=3):
    """
    Bước 1: Lấy danh sách sự kiện user đã đăng ký
    Bước 2: Tính điểm tổng hợp cho MỌI sự kiện chưa đăng ký
            bằng cách cộng dồn similarity với từng sự kiện đã đăng ký
    Bước 3: Gợi ý top_n sự kiện điểm cao nhất
    """
    da_dang_ky = lich_su_dang_ky[student_id]
    
    # Sự kiện chưa đăng ký
    tat_ca = events['event_id'].tolist()
    chua_dang_ky = [e for e in tat_ca if e not in da_dang_ky]
    
    print(f"  ✅ {student_id} đã đăng ký: {da_dang_ky}")
    print(f"  ❓ Chưa đăng ký: {chua_dang_ky}\n")
    
    # Tính điểm cho từng sự kiện chưa đăng ký
    scores = {}
    for chua in chua_dang_ky:
        idx_chua = events[events['event_id'] == chua].index[0]
        total_score = 0
        
        for da in da_dang_ky:
            idx_da = events[events['event_id'] == da].index[0]
            # Cộng dồn similarity: sự kiện chưa đăng ký vs từng sự kiện đã đăng ký
            total_score += cosine_sim[idx_chua][idx_da]
        
        scores[chua] = total_score
    
    # Sắp xếp theo điểm giảm dần
    top_ids = sorted(scores, key=scores.get, reverse=True)[:top_n]
    
    result = events[events['event_id'].isin(top_ids)][
        ['event_id', 'ten_su_kien', 'danh_muc']
    ].copy()
    result['score'] = result['event_id'].map(scores).round(3)
    result = result.sort_values('score', ascending=False)
    
    return result

# Test
print("="*55)
print("📌 Gợi ý cho SV001 (đã đăng ký sự kiện tình nguyện):")
print(goi_y_content_based_cho_user('SV001'))

print("\n" + "="*55)
print("📌 Gợi ý cho SV003 (đã đăng ký sự kiện kỹ năng/học thuật):")
print(goi_y_content_based_cho_user('SV003'))


📌 Gợi ý cho SV001 (đã đăng ký sự kiện tình nguyện):
  ✅ SV001 đã đăng ký: [1, 6]
  ❓ Chưa đăng ký: [2, 3, 4, 5, 7]

   event_id                         ten_su_kien     danh_muc  score
2         3  Tình nguyện dọn vệ sinh môi trường  tinh_nguyen  0.574
1         2                Hội thảo kỹ năng mềm      ky_nang  0.000
3         4              Cuộc thi lập trình ACM    hoc_thuat  0.000

📌 Gợi ý cho SV003 (đã đăng ký sự kiện kỹ năng/học thuật):
  ✅ SV003 đã đăng ký: [2, 4, 5, 7]
  ❓ Chưa đăng ký: [1, 3, 6]

   event_id                         ten_su_kien     danh_muc  score
0         1                Hiến máu tình nguyện  tinh_nguyen    0.0
2         3  Tình nguyện dọn vệ sinh môi trường  tinh_nguyen    0.0
5         6  Chiến dịch hiến máu trường đại học  tinh_nguyen    0.0


In [18]:
print("="*55)
print("📌 Sinh viên xem: Hội thảo kỹ năng mềm (id=2)")
print(goi_y_content_based(event_id=2, top_n=3))

print("\n" + "="*55)
print("📌 Sinh viên xem: Cuộc thi lập trình ACM (id=4)")
print(goi_y_content_based(event_id=4, top_n=3))

📌 Sinh viên xem: Hội thảo kỹ năng mềm (id=2)
   event_id                         ten_su_kien      danh_muc  diem_ren_luyen  \
6         7       Workshop kỹ năng thuyết trình       ky_nang               6   
4         5  Hội thảo định hướng nghề nghiệp IT  huong_nghiep               5   
3         4              Cuộc thi lập trình ACM     hoc_thuat               7   

   similarity_score  
6             0.642  
4             0.154  
3             0.065  

📌 Sinh viên xem: Cuộc thi lập trình ACM (id=4)
   event_id                         ten_su_kien      danh_muc  diem_ren_luyen  \
4         5  Hội thảo định hướng nghề nghiệp IT  huong_nghiep               5   
6         7       Workshop kỹ năng thuyết trình       ky_nang               6   
1         2                Hội thảo kỹ năng mềm       ky_nang               5   

   similarity_score  
4             0.209  
6             0.114  
1             0.065  
